# Module 8 Assignment - RNN Text Classifier Starter Code
---
<div class="alert alert-block alert-success" style="color:black;">
<b>To Begin:</b> Use this <b>Module_Eight_SMS_Starter_Code.ipynb</b> file to complete the assignment. Then, you may create your own .ipynb file or .py file to complete this assignment.
<br><br>
<b>When Finished:</b> When you have completed your work, submit the work as a <b>.ipynb file</b> or a <b>.py</b> file to BrightSpace.
</div>

<div class="alert alert-block alert-info" style="color:black;">
Look for the header <b>Start of Rubric Graded Items</b> to work on your assignment if you are going to submit this file as the work submission.
</div>

<div class="alert alert-block alert-danger" style="color:black;">
<b>GPU/CUDA/Memory Warnings/Errors:</b> You may receive some errors referencing that GPUs will not be used, CUDA could not be found, or free system memory allocation errors. These and a few others, are standard errors that can be ignored here as they are environment based.<br><br>
<b>Example messages:</b>
    <ul>
        <li>Could not find cuda drivers on your machine, GPU will not be used.</li>
        <li>Please check linkage and avoid linking the same target more than once.</li>
        <li>E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)</li>
        <li>Allocation of ######## exceeds 10% of free system memory</li>
    </ul>
</div>

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.layers import TextVectorization
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score

print("TensorFlow version:", tf.__version__)

AUTOTUNE = tf.data.AUTOTUNE


2026-11-12 00:23:24.902119: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/ubuntu/.local/lib/python3.13/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


TensorFlow version: 2.20.0


In [2]:
# -----------------------------
# 1. Load SMS Spam dataset
# -----------------------------
raw = load_dataset("ucirvine/sms_spam")   # single 'train' split
full_ds = raw["train"]                    # 5,574 messages: 'sms' and 'label'

print("Total samples:", len(full_ds))

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/359k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5574 [00:00<?, ? examples/s]

Total samples: 5574


## Start of Graded Rubric Items

### TODO: Apply preprocessing techniques on the dataset.

In [3]:
# TODO: Adjust train/test/validation splits as desired
splits = full_ds.train_test_split(
    test_size=0.2,
    seed=42,
    stratify_by_column="label"
)
train_ds = splits["train"]
test_ds  = splits["test"]

splits2 = train_ds.train_test_split(
    test_size=0.2,
    seed=42,
    stratify_by_column="label"
)
train_ds = splits2["train"]
val_ds   = splits2["test"]

In [4]:
# Convert to plain Python lists and numpy label arrays
train_texts = [str(s) for s in train_ds["sms"]]
val_texts   = [str(s) for s in val_ds["sms"]]
test_texts  = [str(s) for s in test_ds["sms"]]

train_labels = np.array(train_ds["label"], dtype="int32")
val_labels   = np.array(val_ds["label"],   dtype="int32")
test_labels  = np.array(test_ds["label"],  dtype="int32")

print("Train size:", len(train_texts),
      "Val size:", len(val_texts),
      "Test size:", len(test_texts))

Train size: 3567 Val size: 892 Test size: 1115


In [5]:
# Text vectorization
# TODO: Adjust MAX_TOKENS and MAX_LEN to explore effect on embedding / model performance
MAX_TOKENS = 20000
MAX_LEN = 60

vectorizer = TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="int",
    output_sequence_length=MAX_LEN,
)

text_ds = tf.data.Dataset.from_tensor_slices(train_texts).batch(128)
vectorizer.adapt(text_ds)


2026-11-12 00:26:03.963052: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
2026-11-12 00:26:04.245653: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


### TODO: Construct RNN-based model with an embedding layer.

In [6]:
# TODO: Change EMBED_DIM, RNN type (LSTM/GRU/SimpleRNN), number of units, dropout rate
EMBED_DIM = 128

inputs = keras.Input(shape=(), dtype=tf.string, name="text")
x = vectorizer(inputs)
x = layers.Embedding(MAX_TOKENS, EMBED_DIM)(x)
x = layers.Bidirectional(layers.LSTM(64))(x) 
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)  # binary classification (spam/ham)

model = keras.Model(inputs, outputs)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text (InputLayer)               │ (None)                 │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization              │ (None, 60)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 60, 128)        │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,658,945 (10.14 MB)

 Trainable params: 2,658,945 (10.14 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
BATCH_SIZE = 64
EPOCHS = 5

train_tfds = (
    tf.data.Dataset.from_tensor_slices((train_texts, train_labels))
    .shuffle(1000)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_tfds = (
    tf.data.Dataset.from_tensor_slices((val_texts, val_labels))
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

test_tfds = (
    tf.data.Dataset.from_tensor_slices((test_texts, test_labels))
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

### TODO:  Train the model and adjust parameters as needed.

In [8]:
# TODO: Adjust optimizer, learning rate, batch size, epochs to tune performance
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

history = model.fit(
    train_tfds,
    validation_data=val_tfds,
    epochs=EPOCHS,
)


Epoch 1/5
56/56 ━━━━━━━━━━━━━━━━━━━━ 9s 91ms/step - accuracy: 0.8929 - loss: 0.2935 - val_accuracy: 0.9731 - val_loss: 0.1143
Epoch 2/5
56/56 ━━━━━━━━━━━━━━━━━━━━ 5s 82ms/step - accuracy: 0.9832 - loss: 0.0671 - val_accuracy: 0.9888 - val_loss: 0.0522
Epoch 3/5
56/56 ━━━━━━━━━━━━━━━━━━━━ 5s 86ms/step - accuracy: 0.9924 - loss: 0.0291 - val_accuracy: 0.9821 - val_loss: 0.0617
Epoch 4/5
56/56 ━━━━━━━━━━━━━━━━━━━━ 5s 84ms/step - accuracy: 0.9969 - loss: 0.0155 - val_accuracy: 0.9865 - val_loss: 0.0507
Epoch 5/5
56/56 ━━━━━━━━━━━━━━━━━━━━ 5s 83ms/step - accuracy: 0.9983 - loss: 0.0080 - val_accuracy: 0.9877 - val_loss: 0.0580


### TODO: Evaluate model using validation metric

In [10]:
# TODO: Model evaluation method
test_loss, test_acc = model.evaluate(test_tfds)
print(f"Test accuracy: {test_acc:.4f}, Test loss: {test_loss:.4f}")

val_pred_prob = model.predict(val_tfds)
val_pred = (val_pred_prob > 0.5).astype("int32").reshape(-1)

val_labels_all = []
for _, y in val_tfds:
    val_labels_all.extend(y.numpy())
val_labels_all = np.array(val_labels_all, dtype="int32")

# TODO: Determine accuracy score and f1 score
val_acc = accuracy_score(val_labels_all, val_pred)
val_f1 = f1_score(val_labels_all, val_pred)

print(f"Validation accuracy (manual): {val_acc:.4f}")
print(f"Validation F1 score:          {val_f1:.4f}")

18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.9821 - loss: 0.0877
Test accuracy: 0.9821, Test loss: 0.0877
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 45ms/step
Validation accuracy (manual): 0.9877
Validation F1 score:          0.9528


2026-11-12 00:32:46.524768: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


### TODO: Explain choice of RNN type and bidirectionality

#### MARKDOWN cell for above TODO

### TODO: Explain embedding strategy (dimension, vocab size)

#### A Long Short-Term Memory network was selected because it can understand the order of words in SMS messages and help prevent the vanishing gradient problem found in basic recurrent networks. A Bidirectional wrapper was added to the 64-unit LSTM layer so the model could read messages in both forward and backward directions. This helps the model understand the meaning of words based on what comes before and after them. For example, words such as free, claim, and urgent can help identify spam depending on the other words in the message. This model achieved 98.21 percent test accuracy and 98.77 percent validation accuracy.

### TODO: Justify tuning decisions (learning rate, batch size, epochs, dropout)

#### The preprocessing steps used a TextVectorization layer with a vocabulary limit of 20,000 words. This allowed the model to focus on common words while filtering out less useful words. Each message was adjusted to a fixed length of 60 tokens by adding padding or removing extra words. After this step, an Embedding layer with 128 dimensions converted the words into number-based representations that the model could understand. This helped the model recognize words with similar meanings and learn patterns faster, reaching strong results by Epoch 2.

### TODO: Comment on validation loss trends and any adjustments made

#### The model was trained for 5 epochs with a batch size of 64 using the Adam optimizer and binary cross-entropy loss. A batch size of 64 provided a good balance between stable learning and training speed on the CPU. Five epochs were enough for the model to learn without showing signs of overfitting. The validation loss stayed between approximately 0.05 and 0.11. A dropout rate of 0.3 was also added before the final classification layer to help prevent overfitting. This helped control differences between training and validation results. The model reached 99.83 percent training accuracy while maintaining strong validation and test accuracy.

### TODO: Comment on validation loss trends and any adjustments made

####  During the 5 training epochs, the validation loss dropped from 0.1143 in Epoch 1 to 0.0522 in Epoch 2. It remained low and stable, reaching 0.0580 by Epoch 5. Validation accuracy also improved quickly, reaching 98.88 percent in Epoch 2 and ending at 98.77 percent in Epoch 5. The model also achieved a validation F1 score of 0.9528 and a test accuracy of 98.21 percent. Since the validation loss did not show a steady increase compared to the training loss, there were no clear signs of overfitting. These results show that the selected settings and model structure worked well without needing major changes.